# 03 · 量化部署基准（推荐扩展 1：资源优化）

**矩阵**：fp16 / bnb-int8 / bnb-int4(NF4) / GGUF-Q4
**指标**：模型体积、峰值显存、decode tokens/s、首 token 延迟、质量分（同 test 集）

In [ ]:
# [0] GPU 自检：确认拿到 T4，并钉死 T4 兼容参数
import torch
assert torch.cuda.is_available(), "未分配到 GPU！请 运行时->更改运行时类型 选 T4 GPU"
name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {name} | VRAM: {vram:.1f} GB | CC: {torch.cuda.get_device_capability(0)}")
# T4 = sm7.5：无原生 bf16、无 FlashAttention-2 -> 统一 fp16 + sdpa
DTYPE, ATTN = "float16", "sdpa"
print("钉死参数: dtype=%s attn=%s (T4 无 bf16/FA2)" % (DTYPE, ATTN))


In [ ]:
# [1] 依赖对齐
# 关键：不钉 torch（用 Colab 预装 torch，钉旧版会因下架而失败，见 pitfalls P-11）
# 钉死 transformers/trl 兼容矩阵（trl 0.x 与 transformers 5.x 不兼容，见 P-04）
!pip install -q "transformers==4.57.3" "trl==0.15.0" "peft" "accelerate" "bitsandbytes" \
    "datasets" "sentencepiece" "tiktoken" "pandas" "matplotlib"
import torch, transformers, trl
print("torch", torch.__version__, "| cuda", torch.version.cuda)
print("transformers", transformers.__version__, "| trl", trl.__version__)


In [ ]:
# [2] 克隆仓库并把 src 加入 sys.path（单一事实源）
# 注意：Colab 上 `pip install -e .`(PEP660 editable) 常出现装完仍 import 不到的坑，
# 故不用 editable，直接 sys.path 注入 src（依赖已在 [1] 装好）。
import os, subprocess, sys
os.chdir("/content")   # 归位：避免内核 CWD 被历史运行带偏导致嵌套克隆
if not os.path.exists("cs0502-mini-llm"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Zhe-SH-CN/cs0502-mini-llm.git"], check=True)
_src = os.path.abspath("cs0502-mini-llm/src")
if _src not in sys.path:
    sys.path.insert(0, _src)
import minillm
from minillm.config import ExpConfig
from minillm import data as D
from minillm import evaluate as E
from minillm import train as T
from minillm import quantize as Q
from minillm import bench as B
from minillm.record import Recorder
print("minillm", minillm.__version__, "| src:", _src)


In [ ]:
# [3] 挂载 Google Drive，统一结果/数据落盘路径（Colab 文件系统易失，必须落 Drive）
from google.colab import drive
drive.mount("/content/drive")
import os
BASE = "/content/drive/MyDrive/cs0502-mini-llm"
DATA = os.path.join(BASE, "data"); RESULTS = os.path.join(BASE, "results")
os.makedirs(DATA, exist_ok=True); os.makedirs(RESULTS, exist_ok=True)
rec = Recorder(RESULTS)
print("DATA:", DATA, "| RESULTS:", RESULTS)


In [ ]:
# 结果打表辅助
import pandas as pd
def show(df, title=""):
    if title: print("\n==", title, "==")
    from IPython.display import display
    display(df)
    return df


In [ ]:
!pip install -q llama-cpp-python
import pandas as pd, torch as _t


In [ ]:
# 量化矩阵：体积 + 显存 + 吞吐
rows = []
hf = "Qwen/Qwen2.5-0.5B-Instruct"
prompts = [r["question"] for r in D.load_dataset("courseqa", DATA).split("test")[:16]]
variants = [("fp16", lambda: Q.load_fp16(hf, ATTN)),
            ("int8", lambda: Q.load_bnb(hf, 8, ATTN)),
            ("int4", lambda: Q.load_bnb(hf, 4, ATTN))]
for name, loader in variants:
    try:
        m, t = loader()
        m.to("cuda") if next(m.parameters()).device.type != "cuda" else None
        b = B.bench_throughput(m, t, prompts)
        b["variant"] = name
        b["vram_mb"] = round(_t.cuda.max_memory_allocated()/1024**2,1)
        rows.append(b)
        del m; _t.cuda.empty_cache()
    except Exception as e:
        rows.append({"variant": name, "error": str(e)[:120]})
show(pd.DataFrame(rows), "QUANT BENCH (Qwen2.5-0.5B)")
rec.log_run("quant-bench-0.5b", {"model": hf, "variants": [r["variant"] for r in rows]},
            {"rows": rows}, [])


In [ ]:
# 量化后质量对比（int4 vs fp16，同 test 集）
res = {}
for name, bits in (("fp16", None), ("int4", 4)):
    if bits is None:
        m, t = Q.load_fp16(hf, ATTN); m.to("cuda")
    else:
        m, t = Q.load_bnb(hf, 4, ATTN)
    cfg = ExpConfig(run_id=f"quant-quality-{name}", model_key="qwen2.5-0.5b", task="courseqa", dtype=DTYPE, attn=ATTN)
    mm, ps = E.evaluate(m, t, D.load_dataset("courseqa", DATA), "courseqa", cfg)
    res[name] = {k: round(v,4) for k,v in mm.items() if k in ("exact_match","token_f1","overall")}
    rec.log_run(cfg.run_id, cfg.to_dict(), mm, ps)
    del m; _t.cuda.empty_cache()
show(pd.DataFrame(res).T, "QUANT QUALITY")
